In [307]:
# -----------------------------
# 1단계. 기본 집합 및 참여자 데이터 구성
# -----------------------------

# SKU 집합
P = ["P1", "P2", "P3", "P4"]

# SKU 정보
skus = {
    "P1": {"name": "item1"},
    "P2": {"name": "item2"},
    "P3": {"name": "item3"},
    "P4": {"name": "item4"},
}

In [308]:
# -----------------------------
# 1-1. 화주 데이터
# -----------------------------

# inventory: 현재 가용 재고
# ask_price: 화주가 최소한 받고 싶은 SKU 단위 가격
# beta: 화주 쪽 최소 거래비율
#       해당 화주-SKU 재고를 사용하려면 최소 beta 비율 이상 거래되어야 함

sellers = {
    "S1": {
        "name": "seller1",
        "sku_data": {
            "P1": {"inventory": 160, "ask_price": 4000, "beta": 0.20},  # item1
            "P2": {"inventory": 80,  "ask_price": 4200, "beta": 0.20},  # item2
        }
    },

    "S2": {
        "name": "seller2",
        "sku_data": {
            "P1": {"inventory": 200, "ask_price": 4100, "beta": 0.20},  # item1
            "P2": {"inventory": 180, "ask_price": 4300, "beta": 0.20},  # item2
            "P3": {"inventory": 90,  "ask_price": 4400, "beta": 0.20},  # item3
        }
    },

    "S3": {
        "name": "seller3",
        "sku_data": {
            "P3": {"inventory": 180, "ask_price": 4500, "beta": 0.20},  # item3
            "P4": {"inventory": 300, "ask_price": 4100, "beta": 0.10},  # item4
        }
    },

    "S4": {
        "name": "seller4",
        "sku_data": {
            "P2": {"inventory": 120, "ask_price": 4400, "beta": 0.20},  # item2
            "P4": {"inventory": 200, "ask_price": 4200, "beta": 0.10},  # item4
        }
    }
}

In [309]:
# -----------------------------
# 1-2. 수하주 / 구매자 주문 데이터
# -----------------------------

# budget: 주문 전체 예산 상한
# priority: 주문 처리 우선순위
# split_allowed: 여러 화주의 재고를 조합하는 분할 이행 허용 여부
# demand: SKU별 요청 수량
# unit_bid: 수하주가 해당 SKU 1개에 대해 지불할 수 있는 최대 가격
# alpha: SKU별 최소 충족률
#        alpha = 1.0이면 전량 필수
#        0 < alpha < 1이면 최소 충족률 이상 부분 수락 가능
#        alpha = 0이면 선택 품목

buyers = {
    "B1": {
        "name": "buyer1",
        "orders": {
            "O1": {
                "budget": 5000000, "priority": 0.90, "split_allowed": True,
                "items": {
                    "P1": {"demand": 300, "unit_bid": 6000, "alpha": 1.0},
                    "P2": {"demand": 220, "unit_bid": 5800, "alpha": 0.8},
                    "P3": {"demand": 180, "unit_bid": 6200, "alpha": 0.6},
                    "P4": {"demand": 200, "unit_bid": 5400, "alpha": 0.0},
                }
            }
        }
    },

    "B2": {
        "name": "buyer2",
        "orders": {
            "O2": {
                "budget": 3500000, "priority": 0.75, "split_allowed": True,
                "items": {
                    "P1": {"demand": 180, "unit_bid": 5900, "alpha": 1.0},
                    "P3": {"demand": 200, "unit_bid": 6100, "alpha": 0.7},
                    "P4": {"demand": 150, "unit_bid": 5300, "alpha": 0.0},
                }
            }
        }
    },

    "B3": {
        "name": "buyer3",
        "orders": {
            "O3": {
                "budget": 3000000, "priority": 0.60, "split_allowed": True,
                "items": {
                    "P2": {"demand": 250, "unit_bid": 5700, "alpha": 0.8},
                    "P3": {"demand": 150, "unit_bid": 6000, "alpha": 0.6},
                    "P4": {"demand": 200, "unit_bid": 5200, "alpha": 0.0},
                }
            }
        }
    }
}

In [310]:
# -----------------------------
# 1-3. 운송 및 fulfillment 데이터
# -----------------------------

transporters = {
    "K1": {
        "name": "carrier1",
        "capacity": 1000,
        "unit_cost": 500
    },

    "K2": {
        "name": "carrier2",
        "capacity": 700,
        "unit_cost": 650
    }
}

In [311]:
# -----------------------------
# 2단계. 현재 처리할 주문 선택
# -----------------------------

def select_next_order(buyers):
    """
    priority가 가장 높은 주문을 선택한다.
    순차 처리 구조이므로 한 번에 하나의 주문만 처리한다.
    """

    order_pool = []

    for b, bdata in buyers.items():
        for o, odata in bdata["orders"].items():
            order_pool.append((b, o, odata))

    selected_b, selected_o, selected_order = max(
        order_pool,
        key=lambda x: x[2]["priority"]
    )

    return selected_b, selected_o, selected_order


b, o, order = select_next_order(buyers)

print("Selected buyer:", b)
print("Selected order:", o)

Selected buyer: B1
Selected order: O1


In [312]:
# -----------------------------
# 3단계. M 및 거래 가능 tuple 생성
# -----------------------------

def build_valid_tuples(order, sellers, transporters):
    """
    현재 주문에 대해 거래 가능한 (seller, sku, carrier) tuple만 생성한다.

    거래 가능 조건:
    buyer unit_bid >= seller ask_price + carrier unit_cost

    M[(s,p,k)]:
    해당 조합에서 이행 가능한 최대 수량
    """

    valid_tuples = []
    M = {}

    for p, item in order["items"].items():

        demand = item["demand"]
        buyer_bid = item["unit_bid"]

        for s, sdata in sellers.items():

            # 화주 s가 SKU p를 보유하지 않으면 제외
            if p not in sdata["sku_data"]:
                continue

            inventory = sdata["sku_data"][p]["inventory"]
            seller_ask = sdata["sku_data"][p]["ask_price"]

            for k, kdata in transporters.items():

                capacity = kdata["capacity"]
                carrier_cost = kdata["unit_cost"]

                # 거래 가능성 조건
                if buyer_bid >= seller_ask + carrier_cost:

                    m_value = min(demand, inventory, capacity)

                    if m_value > 0:
                        valid_tuples.append((s, p, k))
                        M[(s, p, k)] = m_value

    return valid_tuples, M


valid_tuples, M = build_valid_tuples(order, sellers, transporters)

print("Valid tuples")
for tup in valid_tuples:
    print(tup, "M =", M[tup])

Valid tuples
('S1', 'P1', 'K1') M = 160
('S1', 'P1', 'K2') M = 160
('S2', 'P1', 'K1') M = 200
('S2', 'P1', 'K2') M = 200
('S1', 'P2', 'K1') M = 80
('S1', 'P2', 'K2') M = 80
('S2', 'P2', 'K1') M = 180
('S2', 'P2', 'K2') M = 180
('S4', 'P2', 'K1') M = 120
('S4', 'P2', 'K2') M = 120
('S2', 'P3', 'K1') M = 90
('S2', 'P3', 'K2') M = 90
('S3', 'P3', 'K1') M = 180
('S3', 'P3', 'K2') M = 180
('S3', 'P4', 'K1') M = 200
('S3', 'P4', 'K2') M = 200
('S4', 'P4', 'K1') M = 200
('S4', 'P4', 'K2') M = 200


In [313]:
# -----------------------------
# 4단계. Gurobi 모델 생성
# -----------------------------

import gurobipy as gp
from gurobipy import GRB

model = gp.Model("sequential_many_to_one_matching")

# 현재 주문에 포함된 SKU 집합
P_o = list(order["items"].keys())

# 유효한 화주-SKU pair
valid_seller_sku_pairs = list(
    set((s, p) for (s, p, k) in valid_tuples)
)

# 유효한 운송사 집합
valid_carriers = list(
    set(k for (s, p, k) in valid_tuples)
)

In [314]:
# -----------------------------
# 5단계. 의사결정변수
# -----------------------------

# x: 현재 주문 수락 여부
x = model.addVar(
    vtype=GRB.BINARY,
    name="x_order"
)

# q[s,p,k]: 화주 s의 SKU p를 운송사 k를 통해 이행하는 수량
# 거래 가능한 tuple에 대해서만 생성
q = model.addVars(
    valid_tuples,
    vtype=GRB.CONTINUOUS,
    lb=0,
    name="q"
)

# z[s,p,k]: 해당 화주-SKU-운송사 조합 사용 여부
z = model.addVars(
    valid_tuples,
    vtype=GRB.BINARY,
    name="z"
)

# y[s,p]: 화주 s의 SKU p lot 사용 여부
y = model.addVars(
    valid_seller_sku_pairs,
    vtype=GRB.BINARY,
    name="y"
)

# u[k]: 운송사 k 사용 여부
u = model.addVars(
    valid_carriers,
    vtype=GRB.BINARY,
    name="u"
)

# Q[p]: SKU p의 총 이행 수량
Q = model.addVars(
    P_o,
    vtype=GRB.CONTINUOUS,
    lb=0,
    name="Q"
)

# h[p]: SKU p의 미충족 수량
h = model.addVars(
    P_o,
    vtype=GRB.CONTINUOUS,
    lb=0,
    name="h"
)

In [315]:
# -----------------------------
# 6단계. 목적식
# -----------------------------

# 부족 penalty 설정
# alpha = 0인 선택 품목은 미충족 penalty를 0으로 둔다.
lambda_penalty = {}

for p, item in order["items"].items():
    if item["alpha"] == 0:
        lambda_penalty[p] = 0
    else:
        lambda_penalty[p] = 0.5 * item["unit_bid"]


# 플랫폼 이익
welfare_expr = gp.quicksum(
    (
        order["items"][p]["unit_bid"]
        - sellers[s]["sku_data"][p]["ask_price"]
        - transporters[k]["unit_cost"]
    ) * q[s, p, k]
    for (s, p, k) in valid_tuples
)

# 부분 수락 또는 미충족 penalty
shortage_penalty_expr = gp.quicksum(
    lambda_penalty[p] * h[p]
    for p in P_o
)

model.setObjective(
    welfare_expr - shortage_penalty_expr,
    GRB.MAXIMIZE
)

In [316]:
# -----------------------------
# 7-1. SKU별 총 이행 수량 정의
# -----------------------------

for p in P_o:
    model.addConstr(
        Q[p] == gp.quicksum(
            q[s, p2, k]
            for (s, p2, k) in valid_tuples
            if p2 == p
        ),
        name=f"total_fulfilled_{p}"
    )

In [317]:
# -----------------------------
# 7-2. 최소 충족률 제약
# -----------------------------

for p, item in order["items"].items():
    model.addConstr(
        Q[p] >= item["alpha"] * item["demand"] * x,
        name=f"min_fulfillment_{p}"
    )

In [318]:
# -----------------------------
# 7-3. 수요 초과 이행 방지
# -----------------------------

for p, item in order["items"].items():
    model.addConstr(
        Q[p] <= item["demand"] * x,
        name=f"max_demand_{p}"
    )

In [319]:
# -----------------------------
# 7-4. 부족 수량 정의
# -----------------------------

for p, item in order["items"].items():
    model.addConstr(
        h[p] == item["demand"] * x - Q[p],
        name=f"shortage_{p}"
    )

In [320]:
# -----------------------------
# 7-5. 화주 재고 제약
# -----------------------------

for (s, p) in valid_seller_sku_pairs:
    model.addConstr(
        gp.quicksum(
            q[s2, p2, k]
            for (s2, p2, k) in valid_tuples
            if s2 == s and p2 == p
        )
        <= sellers[s]["sku_data"][p]["inventory"],
        name=f"inventory_{s}_{p}"
    )

In [321]:
# -----------------------------
# 7-6. 운송 capacity 제약
# -----------------------------

for k in valid_carriers:
    model.addConstr(
        gp.quicksum(
            q[s, p, k2]
            for (s, p, k2) in valid_tuples
            if k2 == k
        )
        <= transporters[k]["capacity"] * u[k],
        name=f"capacity_{k}"
    )

In [322]:
# -----------------------------
# 7-7. q-z 연결 제약
# -----------------------------

for (s, p, k) in valid_tuples:
    model.addConstr(
        q[s, p, k] <= M[(s, p, k)] * z[s, p, k],
        name=f"link_q_z_{s}_{p}_{k}"
    )

In [323]:
# -----------------------------
# 7-8. z-y 연결 제약
# -----------------------------

for (s, p, k) in valid_tuples:
    model.addConstr(
        z[s, p, k] <= y[s, p],
        name=f"link_z_y_{s}_{p}_{k}"
    )

In [324]:
# -----------------------------
# 7-9. 화주 최소 거래비율 제약
# -----------------------------

for (s, p) in valid_seller_sku_pairs:

    beta = sellers[s]["sku_data"][p]["beta"]
    inventory = sellers[s]["sku_data"][p]["inventory"]

    model.addConstr(
        gp.quicksum(
            q[s2, p2, k]
            for (s2, p2, k) in valid_tuples
            if s2 == s and p2 == p
        )
        >= beta * inventory * y[s, p],
        name=f"min_trade_ratio_{s}_{p}"
    )

In [325]:
# -----------------------------
# 7-10. z-u 연결 제약
# -----------------------------

for (s, p, k) in valid_tuples:
    model.addConstr(
        z[s, p, k] <= u[k],
        name=f"link_z_u_{s}_{p}_{k}"
    )

In [326]:
# -----------------------------
# 7-11. split_allowed=False 주문의 SKU별 split 금지 제약
# -----------------------------

# split_allowed가 False이면 같은 SKU를 여러 화주/운송사 조합으로 쪼개서 이행하지 못하게 한다.
# split_allowed가 True이면 이 제약을 추가하지 않아 기존처럼 분할 이행을 허용한다.
if not order["split_allowed"]:
    for p in P_o:
        model.addConstr(
            gp.quicksum(
                z[s, p2, k]
                for (s, p2, k) in valid_tuples
                if p2 == p
            ) <= x,
            name=f"no_split_sku_{p}"
        )

In [327]:
# -----------------------------
# 7-12. 구매자 예산 제약: 실제 결제금액 기준
# -----------------------------

# theta: 거래 surplus 중 플랫폼이 가져가는 비율
# theta = 0.5이면 feasible price interval의 중간 가격을 사용한다.
theta = 0.5

# 수하주 실제 지불 단가
# buyer_payment_price = ask_price + carrier_cost + theta * surplus
buyer_payment_price = {}

for (s, p, k) in valid_tuples:
    unit_bid = order["items"][p]["unit_bid"]
    ask_price = sellers[s]["sku_data"][p]["ask_price"]
    carrier_cost = transporters[k]["unit_cost"]
    unit_surplus = unit_bid - ask_price - carrier_cost

    buyer_payment_price[s, p, k] = ask_price + carrier_cost + theta * unit_surplus

model.addConstr(
    gp.quicksum(
        buyer_payment_price[s, p, k] * q[s, p, k]
        for (s, p, k) in valid_tuples
    )
    <= order["budget"] * x,
    name="buyer_budget_actual_payment"
)

<gurobi.Constr *Awaiting Model Update*>

In [328]:
# -----------------------------
# 8단계. 최적화 실행
# -----------------------------

model.optimize()

Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (win64 - Windows 11+.0 (26200.2))

CPU model: AMD Ryzen 9 5900X 12-Core Processor, instruction set [SSE2|AVX|AVX2]
Thread count: 12 physical cores, 24 logical processors, using up to 24 threads

Optimize a model with 91 rows, 56 columns and 241 nonzeros (Max)
Model fingerprint: 0x2c7b1630
Model has 21 linear objective coefficients
Variable types: 26 continuous, 30 integer (30 binary)
Coefficient statistics:
  Matrix range     [1e+00, 5e+06]
  Objective range  [6e+02, 3e+03]
  Bounds range     [1e+00, 1e+00]
  RHS range        [8e+01, 3e+02]

Found heuristic solution: objective -0.0000000
Presolve removed 66 rows and 39 columns
Presolve time: 0.00s
Presolved: 25 rows, 17 columns, 69 nonzeros
Variable types: 10 continuous, 7 integer (7 binary)

Root relaxation: objective 1.049000e+06, 9 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | In

In [329]:
# -----------------------------
# 9-1. 매칭 결과를 records 형태로 저장 + 가격결정
# -----------------------------

matching_records = []

if model.status == GRB.OPTIMAL and x.X > 0.5:

    for (s, p, k) in valid_tuples:
        qty = q[s, p, k].X

        if qty > 1e-6:
            unit_bid = order["items"][p]["unit_bid"]
            ask_price = sellers[s]["sku_data"][p]["ask_price"]
            carrier_cost = transporters[k]["unit_cost"]

            # 단위 social surplus
            unit_surplus = unit_bid - ask_price - carrier_cost

            # -----------------------------
            # 가격결정
            # -----------------------------

            # 수하주 실제 지불 단가
            buyer_unit_payment = buyer_payment_price[s, p, k]

            # 화주 실제 지급 단가
            # 초기 모형에서는 화주 최소 수용가격만 지급한다고 가정
            seller_payment_price = ask_price

            # 플랫폼 단위 이익
            platform_unit_profit = buyer_unit_payment - seller_payment_price - carrier_cost

            # 총액 계산
            buyer_total_payment = buyer_unit_payment * qty
            seller_total_payment = seller_payment_price * qty
            carrier_total_cost = carrier_cost * qty
            platform_total_profit = platform_unit_profit * qty
            total_surplus = unit_surplus * qty

            matching_records.append({
                "buyer_id": b,
                "buyer_name": buyers[b]["name"],
                "order_id": o,

                "seller_id": s,
                "seller_name": sellers[s]["name"],

                "sku": p,
                "item": skus[p]["name"],
                "quantity": qty,

                # bid-ask-cost 정보
                "unit_bid": unit_bid,
                "ask_price": ask_price,
                "carrier_id": k,
                "carrier_name": transporters[k]["name"],
                "unit_cost": carrier_cost,
                "unit_surplus": unit_surplus,

                # 가격결정 결과
                "buyer_payment_price": buyer_unit_payment,
                "seller_payment_price": seller_payment_price,
                "platform_unit_profit": platform_unit_profit,

                # 총액 결과
                "buyer_total_payment": buyer_total_payment,
                "seller_total_payment": seller_total_payment,
                "carrier_total_cost": carrier_total_cost,
                "platform_total_profit": platform_total_profit,
                "total_surplus": total_surplus
            })

In [330]:
# -----------------------------
# 9-2. 화주 x 수하주 매트릭스 출력
# -----------------------------

import pandas as pd
from IPython.display import display, HTML

def build_seller_buyer_matrix(matching_records):
    if len(matching_records) == 0:
        return pd.DataFrame()

    df = pd.DataFrame(matching_records)
    df["buyer_order"] = df["buyer_id"] + "-" + df["order_id"]

    df["cell_text"] = (
        df["sku"] + " (" + df["item"] + ")" +
        "<br>수량: " + df["quantity"].round(2).astype(str) +
        "<br>수하주 bid: " + df["unit_bid"].astype(str) +
        "<br>화주 ask: " + df["ask_price"].astype(str) +
        "<br>운송: " + df["carrier_id"] +
        "<br>운송비: " + df["unit_cost"].astype(str) +
        "<br>단위 surplus: " + df["unit_surplus"].round(2).astype(str) +
        "<br>수하주 지불단가: " + df["buyer_payment_price"].round(2).astype(str) +
        "<br>화주 지급단가: " + df["seller_payment_price"].round(2).astype(str) +
        "<br>플랫폼 단위이익: " + df["platform_unit_profit"].round(2).astype(str)
    )

    grouped = (
        df.groupby(["seller_id", "buyer_order"])["cell_text"]
        .apply(lambda x: "<br><br>".join(x))
        .reset_index()
    )

    matrix = grouped.pivot(
        index="seller_id",
        columns="buyer_order",
        values="cell_text"
    ).fillna("-")

    return matrix


seller_buyer_matrix = build_seller_buyer_matrix(matching_records)

html = seller_buyer_matrix.to_html(escape=False)
display(HTML(html))

buyer_order,B1-O1
seller_id,
S1,P1 (item1)수량: 160.0수하주 bid: 6000화주 ask: 4000운송: K1운송비: 500단위 surplus: 1500수하주 지불단가: 5250.0화주 지급단가: 4000플랫폼 단위이익: 750.0P2 (item2)수량: 80.0수하주 bid: 5800화주 ask: 4200운송: K1운송비: 500단위 surplus: 1100수하주 지불단가: 5250.0화주 지급단가: 4200플랫폼 단위이익: 550.0
S2,P1 (item1)수량: 140.0수하주 bid: 6000화주 ask: 4100운송: K1운송비: 500단위 surplus: 1400수하주 지불단가: 5300.0화주 지급단가: 4100플랫폼 단위이익: 700.0P2 (item2)수량: 140.0수하주 bid: 5800화주 ask: 4300운송: K1운송비: 500단위 surplus: 1000수하주 지불단가: 5300.0화주 지급단가: 4300플랫폼 단위이익: 500.0P3 (item3)수량: 90.0수하주 bid: 6200화주 ask: 4400운송: K1운송비: 500단위 surplus: 1300수하주 지불단가: 5550.0화주 지급단가: 4400플랫폼 단위이익: 650.0
S3,P3 (item3)수량: 90.0수하주 bid: 6200화주 ask: 4500운송: K1운송비: 500단위 surplus: 1200수하주 지불단가: 5600.0화주 지급단가: 4500플랫폼 단위이익: 600.0P4 (item4)수량: 200.0수하주 bid: 5400화주 ask: 4100운송: K1운송비: 500단위 surplus: 800수하주 지불단가: 5000.0화주 지급단가: 4100플랫폼 단위이익: 400.0


In [331]:
# -----------------------------
# 10단계. 재고 및 capacity 업데이트
# -----------------------------

def update_state_after_matching(sellers, transporters, q, valid_tuples):
    """
    현재 주문 처리 결과를 반영하여
    화주 재고와 운송 capacity를 업데이트한다.
    """

    for (s, p, k) in valid_tuples:

        used_qty = q[s, p, k].X

        if used_qty > 1e-6:
            sellers[s]["sku_data"][p]["inventory"] -= used_qty
            transporters[k]["capacity"] -= used_qty

    return sellers, transporters


if model.status == GRB.OPTIMAL and x.X > 0.5:
    sellers, transporters = update_state_after_matching(
        sellers,
        transporters,
        q,
        valid_tuples
    )